# Machine Learning Laboratory Experiments (1 to 10)

This notebook contains the complete step-by-step code implementations for all 10 ML Lab experiments.

## Experiment 1: Exploratory Data Analysis & Data Preprocessing

### Part A: Exploratory Data Analysis (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Step 1: Row & Column Count
df = pd.read_csv('Toyato1a.csv')
print("Step 1 > Shape:", df.shape)

# Step 2: Categorical vs Numerical Columns
num_cols = df.select_dtypes(include='number').columns
cat_cols = df.select_dtypes(include='object').columns
print(f"Step 2 > Numerical: {len(num_cols)}, Categorical: {len(cat_cols)}")

# Step 3: Min, Max, Mean for Numerical Columns
print("Step 3 > Stats:\n", df[num_cols].agg(['min', 'max', 'mean']))

# Step 4: 5-Number Summary & Box Plot for 'Age'
print("Step 4 > 5-Number Summary:\n", df['Age'].describe())
sns.boxplot(x=df['Age'])
plt.title("Step 4 > Age Boxplot")
plt.show()

# Step 5: IQR Outliers for 'Guarantee_Period'
Q1 = df['Guarantee_Period'].quantile(0.25)
Q3 = df['Guarantee_Period'].quantile(0.75)
IQR = Q3 - Q1
iqr_outliers = df[(df['Guarantee_Period'] < Q1 - 1.5*IQR) | (df['Guarantee_Period'] > Q3 + 1.5*IQR)]
print("Step 5 > IQR Outlier Indices:", iqr_outliers.index.tolist())

# Step 6: Z-Score Outliers for 'Guarantee_Period'
z_vals = np.abs(stats.zscore(df['Guarantee_Period'].dropna()))
z_outliers = df.iloc[np.where(z_vals > 3)]
print("Step 6 > Z-Score Outlier Indices:", z_outliers.index.tolist())

# Step 7: Correlation Matrix
features = ['Age', 'Price', 'Quarterly_Tax', 'Weight']
corr = df[features].corr()
print("Step 7 > Correlation Matrix:\n", corr)

# Step 8: High Positive & Negative Pairs
unstacked = corr.unstack()
unstacked = unstacked[unstacked != 1.0]
print("Step 8 > Top Positive Pair:", unstacked.idxmax(), unstacked.max())
print("Step 8 > Top Negative Pair:", unstacked.idxmin(), unstacked.min())

# Step 9: Correlation > 70%
high_corr = unstacked[abs(unstacked) > 0.70]
print("Step 9 > Pairs > 70% Correlation:\n", high_corr)

# Step 10: Skewness Analysis for 'cc'
skewness = df['cc'].skew()
skew_type = "Right Skewed" if skewness > 0.5 else "Left Skewed" if skewness < -0.5 else "No Skew"
print(f"Step 10 > Skewness: {skewness:.2f} ({skew_type})")
sns.histplot(df['cc'], kde=True)
plt.show()

# Step 11: Categorical Univariate Analysis ('Fuel_Type')
sns.countplot(data=df, x='Fuel_Type')
plt.show()

# Step 12: Continuous Univariate Analysis ('Age')
sns.violinplot(y=df['Age'])
sns.swarmplot(y=df['Age'], color='black', alpha=0.5)
plt.show()

# Step 13: Continuous Bivariate Analysis ('Age' vs 'Price')
sns.scatterplot(data=df, x='Age', y='Price')
plt.show()

# Step 14: Categorical vs Continuous Analysis ('Fuel_Type' vs 'Age')
sns.boxplot(data=df, x='Fuel_Type', y='Age')
plt.show()

# Step 15: Categorical vs Categorical Analysis ('Fuel_Type' vs 'Gears')
print("Step 15 > Crosstab:\n", pd.crosstab(df['Fuel_Type'], df['Gears']))
sns.countplot(data=df, x='Fuel_Type', hue='Gears')
plt.show()

# Step 16: Multivariate Pair Plot
sns.pairplot(df[['Age', 'Price', 'KM', 'Weight', 'Fuel_Type']], hue='Fuel_Type')
plt.show()

### Part B: Data Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_b = pd.read_csv('Toyato1b.csv')

# Step 17: Null Value Identification & Percentage
null_pct = (df_b.isnull().sum() / len(df_b)) * 100
print("Step 17 > Missing %:\n", null_pct[null_pct > 0])

# Step 18: Mean Imputation (<10% missing)
print("Step 18 > Before:", df_b['Age'].isnull().sum())
df_b['Age'].fillna(df_b['Age'].mean(), inplace=True)
print("Step 18 > After:", df_b['Age'].isnull().sum())

# Step 19: Nearest Method Imputation (<10% missing)
df_b['KM'] = df_b['KM'].interpolate(method='nearest')

# Step 20: Mode Imputation for Categorical (<10% missing)
df_b['Fuel_Type'].fillna(df_b['Fuel_Type'].mode()[0], inplace=True)

# Step 21: Drop Columns with >10% Missing Values
high_null_cols = null_pct[null_pct > 10].index
df_b.drop(columns=high_null_cols, inplace=True)

# Step 22: Drop Outlier Rows (Z-Score > 3)
z_vals = np.abs(stats.zscore(df_b['Guarantee_Period'].dropna()))
df_b = df_b[z_vals <= 3]

# Step 23 & 24: Handle Full Duplicate Rows
print("Step 23 > Duplicate %:", df_b.duplicated().mean() * 100)
df_b.drop_duplicates(inplace=True)

# Step 25 & 26: Handle Subset Duplicates
subset_cols = ['Price', 'Age', 'Mfg_Month', 'Fuel_Type']
df_b.drop_duplicates(subset=subset_cols, inplace=True)

# Step 27: Min-Max Normalization
df_b['Tax_MinMax'] = (df_b['Quarterly_Tax'] - df_b['Quarterly_Tax'].min()) / (df_b['Quarterly_Tax'].max() - df_b['Quarterly_Tax'].min())

# Step 28: Z-Score Normalization
df_b['Tax_ZScore'] = (df_b['Quarterly_Tax'] - df_b['Quarterly_Tax'].mean()) / df_b['Quarterly_Tax'].std()

# Step 29: Label Encoding
df_b['Fuel_Type_LE'] = LabelEncoder().fit_transform(df_b['Fuel_Type'].astype(str))

# Step 30: One-Hot Encoding
df_b = pd.get_dummies(df_b, columns=['Fuel_Type'])

## Experiment 2: Feature Selection & Extraction

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_regression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from PIL import Image

# Step 1: Read Data
df = pd.read_csv('B3_toyato_feature.csv')
temp_df = df.copy(deep=True)

# Step 2: Drop Specified Columns
temp_df.drop(columns=['BOVAG_Guarantee', 'Doors'], inplace=True, errors='ignore')

# Step 3: Identify & Drop Columns with IQR Outliers
num_cols = temp_df.select_dtypes(include='number').columns
outlier_cols = []
for col in num_cols:
    Q1, Q3 = temp_df[col].quantile(0.25), temp_df[col].quantile(0.75)
    if ((temp_df[col] < Q1 - 1.5*(Q3-Q1)) | (temp_df[col] > Q3 + 1.5*(Q3-Q1))).any():
        outlier_cols.append(col)
temp_df.drop(columns=outlier_cols, inplace=True)

# Step 4: Drop Columns with Variance < 10
var_series = temp_df.select_dtypes(include='number').var()
low_var_cols = var_series[var_series < 10].index
temp_df.drop(columns=low_var_cols, inplace=True)

# Step 5 & 6: Drop Features with Correlation < 0.2 with Target 'Price'
corr_target = temp_df.corr()['Price'].abs()
low_corr_cols = corr_target[corr_target < 0.2].index
temp_df.drop(columns=low_corr_cols, inplace=True)

# Step 7 & 8: Drop One Feature from Highly Correlated Pairs (>80%)
corr_matrix = temp_df.drop(columns=['Price']).corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.80)]
temp_df.drop(columns=to_drop, inplace=True)

# Step 9: Separate X and Y
X = temp_df.drop(columns=['Price'])
Y = temp_df['Price']

# Step 10: SelectKBest (chi2, k=10)
X_positive = X - X.min() + 1
selector = SelectKBest(score_func=chi2, k=min(10, X.shape[1]))
X_kbest = selector.fit_transform(X_positive, Y)

# Step 11: Mutual Information Score
mi_scores = mutual_info_regression(X, Y)
highest_mi_feature = X.columns[np.argmax(mi_scores)]
print("Step 11 > Highest MI Feature:", highest_mi_feature)

# Step 12: Manual PCA from Scratch (First 2 Components)
X_std = (X - X.mean()) / X.std()
cov_matrix = np.cov(X_std.T)
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
sorted_idx = np.argsort(eigenvalues)[::-1]
pca_components = eigenvectors[:, sorted_idx[:2]]
X_pca = np.dot(X_std, pca_components)

# Step 13: LDA using Scikit-Learn
lda = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda.fit_transform(X, pd.qcut(Y, q=3, labels=False))

# Step 14: Read Image & Extract Pixel-by-Pixel Mean
img = Image.open('Lenna.png').convert('L')
img_arr = np.array(img)
print("Step 14 > Pixel Mean:", img_arr.mean())

## Experiment 3: Regression Analysis (From Scratch)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('iris_regression.csv')

# --- Q1: Linear Regression ---
# Step 1 & 2: Read Data & Mean Imputation
df.fillna(df.mean(numeric_only=True), inplace=True)

# Step 3: Select X & Y
X = df['SepalLengthCm'].values
Y = df['SepalWidthCm'].values

# Step 4: Compute b0 and b1
b1 = np.sum((X - X.mean()) * (Y - Y.mean())) / np.sum((X - X.mean())**2)
b0 = Y.mean() - b1 * X.mean()

# Step 5: Equations for Ypred
Y_pred = b0 + b1 * X

# Step 6, 7, 8: Metrics (SSE, RMSE, R2)
SSE = np.sum((Y - Y_pred)**2)
RMSE = np.sqrt(SSE / len(Y))
R2 = 1 - (SSE / np.sum((Y - Y.mean())**2))
print(f"Step 6-8 > SSE: {SSE:.2f}, RMSE: {RMSE:.2f}, R2: {R2:.2f}")

# Step 9: Plot Line
plt.scatter(X, Y)
plt.plot(X, Y_pred, color='red')
plt.title("Simple Linear Regression Fit")
plt.show()

# Step 10: Predict for Given X
print("Step 10 > Prediction for X=5.5:", b0 + b1 * 5.5)

# --- Q2: Multiple Linear Regression ---
# Step 1-4: Select Independent Variables & Calculate Coefficients
X_multi = df[['SepalLengthCm', 'PetalLengthCm']].values
Y_multi = df['SepalWidthCm'].values

X_design = np.hstack((np.ones((X_multi.shape[0], 1)), X_multi))
beta = np.linalg.inv(X_design.T @ X_design) @ X_design.T @ Y_multi

# Step 5-9: Predict, Error Metrics & Custom Prediction
Y_multi_pred = X_design @ beta
sse_m = np.sum((Y_multi - Y_multi_pred)**2)
rmse_m = np.sqrt(sse_m / len(Y_multi))
r2_m = 1 - (sse_m / np.sum((Y_multi - Y_multi.mean())**2))
print(f"Multiple Reg > RMSE: {rmse_m:.2f}, R2: {r2_m:.2f}")

# --- Q3: Logistic Regression ---
# Step 1-6: Gradient Descent Optimization
X_log = df[['SepalWidthCm', 'SepalLengthCm', 'PetalWidthCm', 'PetalLengthCm']].values
Y_log = (df['Species'] == df['Species'].iloc[0]).astype(int).values

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

weights = np.zeros(X_log.shape[1])
bias = 0
lr = 0.01
costs = []

for _ in range(500):
    linear = np.dot(X_log, weights) + bias
    preds = sigmoid(linear)
    cost = -np.mean(Y_log * np.log(preds + 1e-15) + (1 - Y_log) * np.log(1 - preds + 1e-15))
    costs.append(cost)
    
    weights -= lr * np.dot(X_log.T, (preds - Y_log)) / len(Y_log)
    bias -= lr * np.sum(preds - Y_log) / len(Y_log)

plt.plot(costs)
plt.title("Logistic Regression Cost Curve")
plt.xlabel("Iterations")
plt.ylabel("Cost")
plt.show()

## Experiment 4: Single Layer Perceptron (From Scratch)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Step 1 & 2: Read Data & Preprocess
df = pd.read_csv('penguins_SLP.csv')
df.fillna(df.mean(numeric_only=True), inplace=True)

# Step 3: Select X and Y
X = df[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']].values
Y = LabelEncoder().fit_transform(df['species'])
Y = np.where(Y == 0, -1, 1) # Binary perceptron conversion

# Step 4: Standardize Input
X = StandardScaler().fit_transform(X)

# Step 5: Train-Test Split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Step 6: Initialize Parameters
weights = np.zeros(X_train.shape[1])
bias = 0
lr = 0.01
epochs = 100
errors = []

# Step 7: Weight Updates in Training Loop
for _ in range(epochs):
    total_error = 0
    for xi, target in zip(X_train, Y_train):
        update = lr * (target - np.sign(np.dot(xi, weights) + bias))
        weights += update * xi
        bias += update
        total_error += int(update != 0.0)
    errors.append(total_error)

# Step 8: Display Final Weights
print("Step 8 > Final Weights:", weights, "Bias:", bias)

# Step 9: Plot Iteration vs Error
plt.plot(range(1, epochs + 1), errors)
plt.xlabel('Epochs')
plt.ylabel('Errors')
plt.title("SLP Error Convergence")
plt.show()

# Step 10 & 11: Predictions & Evaluation Metrics
Y_pred = np.sign(np.dot(X_test, weights) + bias)
print("Step 11 > Confusion Matrix:\n", confusion_matrix(Y_test, Y_pred))
print("Step 11 > Accuracy:", accuracy_score(Y_test, Y_pred))

## Experiment 5: Multi-Layer Perceptron using PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# Step 1 & 2: Load Dataset
data = load_digits()
X, Y = data.data, data.target

# Step 3: Standardize Features
X = StandardScaler().fit_transform(X)

# Step 4: Split Data
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Step 5: Convert to Tensors
X_train_t = torch.FloatTensor(X_train)
Y_train_t = torch.LongTensor(Y_train)
X_test_t = torch.FloatTensor(X_test)
Y_test_t = torch.LongTensor(Y_test)

# Step 6: Define PyTorch MLP
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 10)
        )
    def forward(self, x):
        return self.network(x)

model = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Step 7: Train Model & Backpropagation
losses = []
for epoch in range(100):
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, Y_train_t)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

# Step 8: Plot Training Curve
plt.plot(losses)
plt.title("MLP Loss Curve")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.show()

# Step 9 & 10: Test & Evaluate Model
with torch.no_grad():
    test_preds = torch.argmax(model(X_test_t), dim=1).numpy()

print("Step 10 > Confusion Matrix:\n", confusion_matrix(Y_test, test_preds))
print("Step 10 > Accuracy:", accuracy_score(Y_test, test_preds))

## Experiment 6: Hierarchical Agglomerative Clustering

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Step 1 & 2: Load & Standardize Data
df = pd.read_csv('customer_clustering.csv').select_dtypes(include='number')
X_scaled = StandardScaler().fit_transform(df)

# Step 3: Linkage & Dendrogram
linked = linkage(X_scaled, method='single', metric='euclidean')
dendrogram(linked)
plt.title("Hierarchical Dendrogram")
plt.show()

# Step 4, 5, 6: Create & Visualize Clustering Models
m1 = AgglomerativeClustering(n_clusters=2, metric='euclidean', linkage='ward')
m2 = AgglomerativeClustering(n_clusters=3, metric='euclidean', linkage='ward')

l1, l2 = m1.fit_predict(X_scaled), m2.fit_predict(X_scaled)
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=l1)
plt.title("Agglomerative Clusters (k=2)")
plt.show()

# Step 7 & 8: Performance Evaluation
s1, s2 = silhouette_score(X_scaled, l1), silhouette_score(X_scaled, l2)
print(f"Step 7 > Silhouette Scores -> Model 1 (k=2): {s1:.2f}, Model 2 (k=3): {s2:.2f}")

# Step 9: Select Best Model & Export to CSV for Tableau
df['Cluster_Label'] = l1 if s1 > s2 else l2
df.to_csv('HAC_Tableau_Export.csv', index=False)

## Experiment 7: Fuzzy C-Means Clustering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from fcmeans import FCM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Step 1, 2, 3: Load, Impute & Standardize
df = pd.read_csv('driverFCM.csv').select_dtypes(include='number')
df.fillna(df.mean(), inplace=True)
X_scaled = StandardScaler().fit_transform(df)

# Step 4 & 5: Initialization & Feature Plot
plt.scatter(X_scaled[:, 0], X_scaled[:, 1])
plt.title("Unclustered Feature Scatter")
plt.show()

# Step 6 & 7: FCM Loop (k=2 to 5) & Silhouette Score Curve
sil_scores = []
fcm_models = []

for k in range(2, 6):
    fcm = FCM(n_clusters=k, m=2, max_iter=150)
    fcm.fit(X_scaled)
    labels = fcm.predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels))
    fcm_models.append((fcm, labels))

plt.plot(range(2, 6), sil_scores)
plt.title("FCM Silhouette Curve")
plt.xlabel("Clusters (k)")
plt.ylabel("Silhouette Score")
plt.show()

# Step 8, 9, 10, 11: Optimal Model & Centroids
best_idx = np.argmax(sil_scores)
best_fcm, best_labels = fcm_models[best_idx]
print("Step 11 > Cluster Centroids:\n", best_fcm.centers)

# Step 12: Export for Tableau
df['FCM_Cluster'] = best_labels
df.to_csv('FCM_Tableau_Export.csv', index=False)

## Experiment 8: Self-Organizing Maps (SOM)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from minisom import MiniSom
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Step 1, 2, 3: Load, Impute & Standardize
df = pd.read_csv('wineFCM.csv').select_dtypes(include='number')
df.fillna(df.mean(), inplace=True)
X_scaled = StandardScaler().fit_transform(df)

# Step 4 & 5: Feature Plot
plt.scatter(X_scaled[:, 0], X_scaled[:, 1])
plt.title("Feature Data Plot")
plt.show()

# Step 6, 7, 8: SOM Loop (k=2 to 5)
scores = []
for k in range(2, 6):
    som = MiniSom(x=1, y=k, input_len=X_scaled.shape[1], sigma=1.0, learning_rate=0.5)
    som.random_weights_init(X_scaled)
    som.train_random(X_scaled, num_iteration=100)
    labels = [np.argmin([np.linalg.norm(x - w) for w in som.get_weights().reshape(-1, X_scaled.shape[1])]) for x in X_scaled]
    scores.append(silhouette_score(X_scaled, labels))

# Step 9, 10, 11: Best Model & Plot
plt.plot(range(2, 6), scores)
plt.title("SOM Silhouette Curve")
plt.xlabel("Grid Dimensions (1xk)")
plt.ylabel("Silhouette Score")
plt.show()

# Step 12: Export for Power BI
df['SOM_Cluster'] = labels
df.to_csv('SOM_PowerBI_Export.csv', index=False)

## Experiment 9: Association Rule Learning (Apriori From Scratch)

In [ ]:
import pandas as pd
from itertools import combinations

# Step 1: Read Data
df = pd.read_csv('Shop1.csv')
transactions = df.dropna().values.tolist()

min_support = 0.08
min_confidence = 0.50

# Step 2: Generate Candidate & Frequent Itemsets Iteratively
def get_frequent_itemsets(transactions, min_sup):
    counts = {}
    for t in transactions:
        for item in t:
            if pd.notna(item):
                counts[frozenset([item])] = counts.get(frozenset([item]), 0) + 1
    
    num_trans = len(transactions)
    freq_sets = {k: v/num_trans for k, v in counts.items() if v/num_trans >= min_sup}
    return freq_sets

freq_itemsets = get_frequent_itemsets(transactions, min_support)
print("Step 2 > Frequent Itemsets:\n", freq_itemsets)

# Step 3 & 4: Generate Association Rules & Filter Confidence
rules = []
for itemset in freq_itemsets:
    if len(itemset) > 1:
        for antecedent in map(frozenset, combinations(itemset, 1)):
            consequent = itemset - antecedent
            conf = freq_itemsets[itemset] / freq_itemsets.get(antecedent, 1)
            if conf >= min_confidence:
                rules.append((antecedent, consequent, conf))

print("Step 3 & 4 > Association Rules (High Confidence):\n", rules)

## Experiment 10: Ensemble Learning (Random Forest & AdaBoost)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# Step 1 & 2: Read Data & Preprocessing
df = pd.read_csv('iris_SLP.csv')
df.fillna(df.mean(numeric_only=True), inplace=True)

# Step 3: Choose X and Y
X = df[['SepalWidthCm', 'SepalLengthCm', 'PetalWidthCm', 'PetalLengthCm']]
Y = LabelEncoder().fit_transform(df['Species'])

# Step 4: Split Data
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# --- Q1: Random Forest ---
print("--- Random Forest Evaluation ---")
for n in [10, 50, 100]:
    for d in [3, 5, None]:
        rf = RandomForestClassifier(n_estimators=n, max_depth=d, random_state=42)
        rf.fit(X_train, Y_train)
        preds = rf.predict(X_test)
        print(f"RF (n={n}, d={d}) -> Accuracy: {accuracy_score(Y_test, preds):.2f}")
        print("Confusion Matrix:\n", confusion_matrix(Y_test, preds))

# --- Q2: AdaBoost ---
print("\n--- AdaBoost Evaluation ---")
for n in [10, 50, 100]:
    for lr in [0.01, 0.1, 1.0]:
        ada = AdaBoostClassifier(n_estimators=n, learning_rate=lr, random_state=42)
        ada.fit(X_train, Y_train)
        preds = ada.predict(X_test)
        print(f"AdaBoost (n={n}, lr={lr}) -> Accuracy: {accuracy_score(Y_test, preds):.2f}")
        print("Confusion Matrix:\n", confusion_matrix(Y_test, preds))